In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (4005, 24)

Columns:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'title_clean', 'topics_clean', 'description_clean', 'transcript_clean', 'combined_text']


In [4]:
# Experiment 1: Title Only

tfidf_title = TfidfVectorizer(stop_words='english')

title_matrix = tfidf_title.fit_transform(df['title_clean'])

title_similarity = cosine_similarity(title_matrix)

print("Title similarity matrix shape:", title_similarity.shape)

Title similarity matrix shape: (4005, 4005)


In [5]:
# Experiment 2: Title + Tags

df['title_tags'] = (
    df['title_clean'] + ' ' +
    df['topics_clean']
)

tfidf_title_tags = TfidfVectorizer(stop_words='english')

title_tags_matrix = tfidf_title_tags.fit_transform(df['title_tags'])

title_tags_similarity = cosine_similarity(title_tags_matrix)

print("Title + Tags similarity matrix shape:", title_tags_similarity.shape)

Title + Tags similarity matrix shape: (4005, 4005)


In [6]:
# Experiment 3: Title + Description + Tags

df['title_description_tags'] = (
    df['title_clean'] + ' ' +
    df['description_clean'] + ' ' +
    df['topics_clean']
)

tfidf_title_description_tags = TfidfVectorizer(stop_words='english')

title_description_tags_matrix = tfidf_title_description_tags.fit_transform(
    df['title_description_tags']
)

title_description_tags_similarity = cosine_similarity(
    title_description_tags_matrix
)

print("Title + Description + Tags similarity matrix shape:",
      title_description_tags_similarity.shape)

Title + Description + Tags similarity matrix shape: (4005, 4005)


In [7]:
# Experiment 4: Title + Description + Transcript + Tags

df['title_description_transcript_tags'] = (
    df['title_clean'] + ' ' +
    df['description_clean'] + ' ' +
    df['transcript_clean'] + ' ' +
    df['topics_clean']
)

tfidf_all = TfidfVectorizer(stop_words='english')

all_features_matrix = tfidf_all.fit_transform(
    df['title_description_transcript_tags']
)

all_features_similarity = cosine_similarity(all_features_matrix)

print("Title + Description + Transcript + Tags similarity matrix shape:",
      all_features_similarity.shape)

Title + Description + Transcript + Tags similarity matrix shape: (4005, 4005)


In [8]:
# Select Five TED Talks for Recommendation Evaluation

selected_talks = df.sample(n=5, random_state=42)

print(selected_talks[['talk_id', 'title']].to_string(index=False))

 talk_id                                           title
     995   Glorious visions in animation and performance
    1307                     Medicine for the 99 percent
    1879       What is so special about the human brain?
   41223                  The mysterious science of pain
   32269 The uncomplicated truth about women's sexuality


In [10]:
# Activity 2: Generate Top 5 Recommendations for Each Selected TED Talk

def get_recommendations(talk_id, similarity_matrix, top_n=5):
    talk_index = df.index[df['talk_id'] == talk_id][0]

    similarity_scores = list(enumerate(similarity_matrix[talk_index]))

    similarity_scores = sorted(
        similarity_scores, key=lambda x: x[1], reverse=True
    )

    recommendations = [
        (df.iloc[index]['title'], score)
        for index, score in similarity_scores
        if index != talk_index
    ][:top_n]

    return recommendations


similarity_matrices = {
    'Experiment 1 - Title Only': title_similarity,
    'Experiment 2 - Title + Tags': title_tags_similarity,
    'Experiment 3 - Title + Description + Tags': title_description_tags_similarity,
    'Experiment 4 - Title + Description + Transcript + Tags': all_features_similarity
}

for _, talk in selected_talks.iterrows():
    print("\nSelected TED Talk:", talk['title'])

    for experiment, matrix in similarity_matrices.items():
        print("\n", experiment)

        recommendations = get_recommendations(talk['talk_id'], matrix)

        for title, score in recommendations:
            print(f"{title} (Similarity: {score:.4f})")


Selected TED Talk: Glorious visions in animation and performance

 Experiment 1 - Title Only
A performance of "Mathemagic" (Similarity: 0.3252)
Suspended animation is within our grasp (Similarity: 0.2783)
Visions of Africa's future, from African filmmakers (Similarity: 0.2589)
An electrifying acoustic guitar performance (Similarity: 0.2309)
Averting the climate crisis (Similarity: 0.0000)

 Experiment 2 - Title + Tags
"Chasms" (Similarity: 0.3635)
"Wild Women" (Similarity: 0.3389)
A multimedia theatrical adventure (Similarity: 0.3334)
Kabuki: The people's dramatic art (Similarity: 0.3164)
If I should have a daughter ... (Similarity: 0.3106)

 Experiment 3 - Title + Description + Tags
A TED speaker's worst nightmare (Similarity: 0.1180)
Improvising on piano, aged 14 (Similarity: 0.1106)
A one-man musical phenomenon (Similarity: 0.1077)
"What You've Got" (Similarity: 0.1074)
"Autopilote" / "Pumper" (Similarity: 0.1012)

 Experiment 4 - Title + Description + Transcript + Tags
Music and e

In [12]:
# Activity 2: Compare Recommendations Across Four Experiments

for talk in selected_talks['title']:

    print("\nSelected TED Talk:", talk)

    talk_results = comparison_df[
        comparison_df['Selected TED Talk'] == talk
    ]

    comparison_table = talk_results.pivot(
        index='Rank',
        columns='Experiment',
        values='Recommended TED Talk'
    )

    display(comparison_table)


Selected TED Talk: Glorious visions in animation and performance


Experiment,Experiment 1 - Title Only,Experiment 2 - Title + Tags,Experiment 3 - Title + Description + Tags,Experiment 4 - Title + Description + Transcript + Tags
Rank,,,,
1,"A performance of ""Mathemagic""","""Chasms""",A TED speaker's worst nightmare,Music and emotion through time
2,Suspended animation is within our grasp,"""Wild Women""","Improvising on piano, aged 14",How to build a thriving music scene in your city
3,"Visions of Africa's future, from African filmm...",A multimedia theatrical adventure,A one-man musical phenomenon,The princess who rewrote history
4,An electrifying acoustic guitar performance,Kabuki: The people's dramatic art,"""What You've Got""",How architecture helped music evolve
5,Averting the climate crisis,If I should have a daughter ...,"""Autopilote"" / ""Pumper""",The DIY orchestra of the future



Selected TED Talk: Medicine for the 99 percent


Experiment,Experiment 1 - Title Only,Experiment 2 - Title + Tags,Experiment 3 - Title + Description + Tags,Experiment 4 - Title + Description + Transcript + Tags
Rank,,,,
1,Between music and medicine,I got 99 problems ... palsy is just one,The troubling reason why vaccines are made too...,"Pool medical patents, save lives"
2,How do we heal medicine?,A new way to fund health care for the most vul...,Body parts on a chip,Why are drug prices so high? Investigating the...
3,Print your own medicine,The coming crisis in antibiotics,"The ""bottom billion""",How to buy happiness
4,I got 99 problems ... palsy is just one,How do we heal medicine?,A new way to fund health care for the most vul...,What if we paid doctors to keep people healthy?
5,Medicine's future? There's an app for that,Crowdsource your health,This virtual lab will revolutionize science class,A bold new way to fund drug research



Selected TED Talk: What is so special about the human brain?


Experiment,Experiment 1 - Title Only,Experiment 2 - Title + Tags,Experiment 3 - Title + Description + Tags,Experiment 4 - Title + Description + Transcript + Tags
Rank,,,,
1,What makes you special?,Re-engineering the brain,Brain-to-brain communication has arrived. How ...,What percentage of your brain do you use?
2,Are you human?,Brain-to-brain communication has arrived. How ...,A neural portrait of the human mind,The neurons that shaped civilization
3,What was so special about Viking ships?,How menopause affects the brain,The real reason for brains,A brain in a supercomputer
4,Special Olympics let me be myself -- a champion,What happens in your brain when you taste food,3 clues to understanding your brain,What investigating neural pathways can reveal ...
5,The brain in love,What percentage of your brain do you use?,The mysterious workings of the adolescent brain,What we'll learn about the brain in the next c...



Selected TED Talk: The mysterious science of pain


Experiment,Experiment 1 - Title Only,Experiment 2 - Title + Tags,Experiment 3 - Title + Description + Tags,Experiment 4 - Title + Description + Transcript + Tags
Rank,,,,
1,"The beautiful, mysterious science of how you hear",How does your brain respond to pain?,How does your brain respond to pain?,How does your brain respond to pain?
2,Would you opt for a life with no pain?,The mystery of chronic pain,The mystery of chronic pain,What surviving the Columbine shooting taught m...
3,The world's most mysterious book,How do pain relievers work?,The amazing story of the man who gave us moder...,How do pain relievers work?
4,How do pain relievers work?,Is marijuana bad for your brain?,How do pain relievers work?,The mystery of chronic pain
5,The most mysterious star in the universe,How stress affects your brain,What surviving the Columbine shooting taught m...,A look inside the brain in real time



Selected TED Talk: The uncomplicated truth about women's sexuality


Experiment,Experiment 1 - Title Only,Experiment 2 - Title + Tags,Experiment 3 - Title + Description + Tags,Experiment 4 - Title + Description + Transcript + Tags
Rank,,,,
1,Why kids need to learn about gender and sexuality,The science of sex,Women should represent women in media,The lies we tell pregnant women
2,The other inconvenient truth,The truth about unwanted arousal,The lies we tell pregnant women,Why medicine often has dangerous side effects ...
3,Women should represent women in media,The virginity fraud,Why we have too few women leaders,His and hers ... health care
4,"Beauty, truth and ... physics?",Women should represent women in media,"Radical women, embracing tradition",The science of sex
5,How to see past your own perspective and find ...,"How to have a healthier, positive relationship...","The keys to a happier, healthier sex life",The political progress women have made -- and ...



## Activity 3: Recommendation Quality Evaluation

### 1. Relevance of Recommended Talks
Experiment 1 (Title Only) generated some relevant recommendations, but also included irrelevant talks. Experiments 2, 3 and 4 generated recommendations with better topic relevance.

### 2. Similarity of Topics and Concepts
Recommendations from Experiments 2, 3 and 4 generally discussed topics or concepts related to the selected TED Talks.

### 3. Irrelevant Recommendations
Some irrelevant recommendations were generated, especially in Experiment 1, because similarity was based mainly on matching words in titles.

### 4. Effect of Adding Transcript
Adding the transcript improved recommendations in several cases by considering the detailed content and concepts discussed in the TED Talks.

### 5. Most Meaningful Feature Combination
Experiment 4 (Title + Description + Transcript + Tags) produced meaningful recommendations in several cases because it considered both the topics and detailed content of the talks.



## Activity 4: Finalize Feature Selection

### Selected Feature Combination
Title + Description + Transcript + Tags (Experiment 4)

### Justification
I selected this feature combination because it considers multiple aspects of a TED Talk. The title provides the main topic, tags identify related subjects, the description provides an overview, and the transcript contains detailed information about the talk.

Combining these features helps the recommendation system identify similarities based on both topics and detailed content, producing more meaningful recommendations.
